# Phase 4 — Elo rating baseline

This self-contained notebook implements and validates MatchCast's chronological Elo baseline. It reads the deterministic Phase 3 match table, records pre-match features before any update, writes `data/processed/matches_with_elo.csv`, and evaluates a chronological holdout.

Configuration: unseen teams start at 1500; K=20; non-neutral home advantage=100 Elo points. For a three-way baseline, a fixed 25% draw prior is used and the remaining 75% is divided according to the Elo expected score. This deliberately simple, untuned mapping avoids learning from the holdout.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_PATH = ROOT / 'data' / 'processed' / 'matches.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'

INITIAL_RATING = 1500.0
K_FACTOR = 20.0
HOME_ADVANTAGE = 100.0
DRAW_PRIOR = 0.25
HOLDOUT_FRACTION = 0.20

matches = pd.read_csv(INPUT_PATH, parse_dates=['date'])
required = {'match_id', 'date', 'home_team', 'away_team', 'neutral', 'result'}
assert required <= set(matches.columns)
assert matches['date'].notna().all() and matches['match_id'].is_unique
matches.shape

(49493, 15)

## Elo implementation and leakage policy

The expected-score formula is `1 / (1 + 10 ** (-difference / 400))`. Features are captured from the rating state at the start of each date. All matches on that date calculate deltas from that frozen state; accumulated deltas are applied only after every row for the date is recorded. Therefore same-day row order cannot alter features, unseen teams deterministically receive 1500, and no match outcome affects its own features.

In [2]:
def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10.0 ** (-(rating_a - rating_b) / 400.0))

def outcome_probabilities(home_elo: float, away_elo: float, neutral: bool) -> tuple[float, float, float]:
    advantage = 0.0 if bool(neutral) else HOME_ADVANTAGE
    home_expectation = expected_score(home_elo + advantage, away_elo)
    return ((1.0 - DRAW_PRIOR) * home_expectation, DRAW_PRIOR,
            (1.0 - DRAW_PRIOR) * (1.0 - home_expectation))

def actual_home_score(result: str) -> float:
    return {'H': 1.0, 'D': 0.5, 'A': 0.0}[result]

def add_elo_features(frame: pd.DataFrame) -> pd.DataFrame:
    ordered = frame.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True).copy()
    ratings: dict[str, float] = {}
    records: list[dict[str, float]] = []
    for _, day in ordered.groupby('date', sort=True):
        start = ratings.copy()
        deltas: dict[str, float] = {}
        for row in day.itertuples(index=False):
            home_elo = start.get(row.home_team, INITIAL_RATING)
            away_elo = start.get(row.away_team, INITIAL_RATING)
            advantage = 0.0 if bool(row.neutral) else HOME_ADVANTAGE
            adjusted_difference = home_elo + advantage - away_elo
            expected_home = expected_score(home_elo + advantage, away_elo)
            p_home, p_draw, p_away = outcome_probabilities(home_elo, away_elo, row.neutral)
            records.append({'home_elo': home_elo, 'away_elo': away_elo,
                            'elo_difference': home_elo - away_elo,
                            'elo_adjusted_difference': adjusted_difference,
                            'elo_home_probability': p_home, 'elo_draw_probability': p_draw,
                            'elo_away_probability': p_away})
            change = K_FACTOR * (actual_home_score(row.result) - expected_home)
            deltas[row.home_team] = deltas.get(row.home_team, 0.0) + change
            deltas[row.away_team] = deltas.get(row.away_team, 0.0) - change
        for team, delta in deltas.items():
            ratings[team] = start.get(team, INITIAL_RATING) + delta
    return pd.concat([ordered, pd.DataFrame(records)], axis=1)

featured = add_elo_features(matches)
featured.to_csv(OUTPUT_PATH, index=False, date_format='%Y-%m-%d')
featured[['date', 'home_team', 'away_team', 'home_elo', 'away_elo', 'elo_difference']].head()

,date,home_team,away_team,home_elo,away_elo,elo_difference
0,1872-11-30,Scotland,England,1500.000000,1500.000000,0.000000
1,1873-03-08,England,Scotland,1502.801300,1497.198700,5.602600
2,1874-03-07,Scotland,England,1490.147921,1509.852079,-19.704159
3,1875-03-06,England,Scotland,1502.122893,1497.877107,4.245787
4,1876-03-04,Scotland,England,1500.790631,1499.209369,1.581262


## Embedded unit and leakage validation

These checks cover expected scores, updates, venue behavior, unseen teams, probability normalization, deterministic output, same-day order independence, and future-outcome leakage.

In [3]:
assert math.isclose(expected_score(1500, 1500), 0.5)
assert expected_score(1600, 1500) > 0.5
neutral_probs = outcome_probabilities(1500, 1500, True)
home_probs = outcome_probabilities(1500, 1500, False)
assert math.isclose(sum(neutral_probs), 1.0) and neutral_probs[0] == neutral_probs[2]
assert home_probs[0] > neutral_probs[0]

fixture = pd.DataFrame([
    {'match_id':'a', 'date':pd.Timestamp('2020-01-01'), 'home_team':'New A', 'away_team':'New B', 'neutral':True, 'result':'H'},
    {'match_id':'b', 'date':pd.Timestamp('2020-01-01'), 'home_team':'New A', 'away_team':'New C', 'neutral':True, 'result':'A'},
    {'match_id':'c', 'date':pd.Timestamp('2020-01-02'), 'home_team':'New A', 'away_team':'New B', 'neutral':True, 'result':'D'},
])
fx = add_elo_features(fixture)
assert (fx.loc[:1, ['home_elo', 'away_elo']] == INITIAL_RATING).all().all()
assert fx.loc[2, 'home_elo'] == INITIAL_RATING  # equal/opposite same-day changes cancel
assert fx.loc[2, 'away_elo'] < INITIAL_RATING

# Reversing same-day source order leaves match-level pre-match features unchanged.
rev = add_elo_features(pd.concat([fixture.iloc[[1, 0]], fixture.iloc[[2]]], ignore_index=True))
cols = ['match_id', 'home_elo', 'away_elo', 'elo_difference']
pd.testing.assert_frame_equal(fx[cols].sort_values('match_id').reset_index(drop=True),
                              rev[cols].sort_values('match_id').reset_index(drop=True))

# Changing a future result cannot change any earlier feature.
changed = fixture.copy(); changed.loc[2, 'result'] = 'H'
before = add_elo_features(fixture).loc[:2, cols]
after = add_elo_features(changed).loc[:2, cols]
pd.testing.assert_frame_equal(before, after)

prob_cols = ['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']
assert np.isfinite(featured[prob_cols]).all().all()
assert ((featured[prob_cols] >= 0) & (featured[prob_cols] <= 1)).all().all()
assert np.allclose(featured[prob_cols].sum(axis=1), 1.0)
pd.testing.assert_frame_equal(featured, add_elo_features(matches))
assert OUTPUT_PATH.exists()
print(f'All Elo validations passed for {len(featured):,} matches.')

All Elo validations passed for 49,493 matches.


## Chronological holdout evaluation

The final 20% of matches by deterministic chronological order is the preliminary holdout. Elo state is updated sequentially across the full history, but every prediction is made before its match result is observed. Metrics are multiclass log loss, multiclass Brier score, and accuracy.

In [4]:
split_index = int(len(featured) * (1.0 - HOLDOUT_FRACTION))
holdout = featured.iloc[split_index:].copy()
assert featured.iloc[split_index - 1]['date'] <= holdout.iloc[0]['date']
class_order = ['H', 'D', 'A']
probs = holdout[prob_cols].to_numpy(float)
actual = pd.get_dummies(pd.Categorical(holdout['result'], categories=class_order)).to_numpy(float)
chosen = probs[np.arange(len(holdout)), np.argmax(actual, axis=1)]
log_loss = float(-np.log(np.clip(chosen, 1e-15, 1.0)).mean())
brier = float(np.square(probs - actual).sum(axis=1).mean())
predicted = np.array(class_order)[np.argmax(probs, axis=1)]
accuracy = float((predicted == holdout['result'].to_numpy()).mean())
summary = pd.Series({
    'total_matches': len(featured), 'holdout_matches': len(holdout),
    'holdout_start': holdout['date'].min().date().isoformat(),
    'holdout_end': holdout['date'].max().date().isoformat(),
    'multiclass_log_loss': log_loss, 'multiclass_brier_score': brier,
    'accuracy': accuracy,
})
assert len(holdout) > 0 and np.isfinite([log_loss, brier, accuracy]).all()
summary

total_matches                  49493
holdout_matches                 9899
holdout_start             2016-03-29
holdout_end               2026-07-03
multiclass_log_loss         0.914493
multiclass_brier_score      0.538621
accuracy                    0.592686
dtype: object

## Limitations

The fixed K-factor ignores match importance and margin of victory. Ratings never regress toward the mean during inactivity. The fixed draw prior is intentionally uncalibrated and cannot respond to team strength or match context. The holdout is a preliminary future-only diagnostic rather than the expanding-window tournament backtest planned for Phase 7.